# Baseline: point cloud semantic segmentation with 3D-GCN (LiDAR only)

Trains the 3D-GCN deformable-kernel segmentation network on SemanticKITTI using only the raw LiDAR
coordinates (x, y, z). This is the initial model and the baseline for the point-painting experiment in
`03_train_gcn3d_rgb_painting.ipynb` (validation mIoU 0.595 in the report).

- **Train sequences:** 00–04, 06, 07, 09, 10. **Validation:** 08. **Test:** 05.
- Only points that project inside the left colour camera image are used; 10,000 are sampled per scan.
- lr = 0.001, batch size 8, Adam, cross-entropy loss.

In [ ]:
from comet_ml import Experiment
import yaml
import glob
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import cv2
import os
import numpy as np
from collections import namedtuple

In [ ]:
# ---- Paths: edit these (or set the environment variables) to match your setup ----
# SemanticKITTI sequences folder containing <seq>/velodyne, <seq>/labels and <seq>/image_2
DATA_ROOT = os.environ.get("SEMKITTI_ROOT", "../dataset/sequences/")
# KITTI odometry calibration folder containing <seq>/calib.txt (with the Tr velodyne->camera matrix)
CALIB_ROOT = os.environ.get("KITTI_CALIB_ROOT", "../dataset/data_odometry_calib/dataset/sequences/")
CONFIG_PATH = "../config/sem_kitti.yaml"
OUTPUT_DIR = "../outputs/"

os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)

## Calibration helpers
These build the LiDAR → camera-2 projection from the KITTI odometry `calib.txt`.

In [ ]:
def load_calib(sequence_path):
        """Load and compute intrinsic and extrinsic cali
        bration parameters."""
        # We'll build the calibration parameters as a dictionary, then
        # convert it to a namedtuple to prevent it from being modified later
        data = {}

        # Load the calibration file
        calib_filepath = os.path.join(sequence_path, 'calib.txt')
        filedata = read_calib_file(calib_filepath)

        # Create 3x4 projection matrices
        P_rect_00 = np.reshape(filedata['P0'], (3, 4))
        P_rect_10 = np.reshape(filedata['P1'], (3, 4))
        P_rect_20 = np.reshape(filedata['P2'], (3, 4))
        P_rect_30 = np.reshape(filedata['P3'], (3, 4))

        data['P_rect_00'] = P_rect_00
        data['P_rect_10'] = P_rect_10
        data['P_rect_20'] = P_rect_20
        data['P_rect_30'] = P_rect_30

        # Compute the rectified extrinsics from cam0 to camN
        T1 = np.eye(4)
        T1[0, 3] = P_rect_10[0, 3] / P_rect_10[0, 0]
        T2 = np.eye(4)
        T2[0, 3] = P_rect_20[0, 3] / P_rect_20[0, 0]
        T3 = np.eye(4)
        T3[0, 3] = P_rect_30[0, 3] / P_rect_30[0, 0]

        # Compute the velodyne to rectified camera coordinate transforms
        data['T_cam0_velo'] = np.reshape(filedata['Tr'], (3, 4))
        data['T_cam0_velo'] = np.vstack([data['T_cam0_velo'], [0, 0, 0, 1]])
        data['T_cam1_velo'] = T1.dot(data['T_cam0_velo'])
        data['T_cam2_velo'] = T2.dot(data['T_cam0_velo'])
        data['T_cam3_velo'] = T3.dot(data['T_cam0_velo'])

        # Compute the camera intrinsics
        data['K_cam0'] = P_rect_00[0:3, 0:3]
        data['K_cam1'] = P_rect_10[0:3, 0:3]
        data['K_cam2'] = P_rect_20[0:3, 0:3]
        data['K_cam3'] = P_rect_30[0:3, 0:3]

        # Compute the stereo baselines in meters by projecting the origin of
        # each camera frame into the velodyne frame and computing the distances
        # between them
        p_cam = np.array([0, 0, 0, 1])
        p_velo0 = np.linalg.inv(data['T_cam0_velo']).dot(p_cam)
        p_velo1 = np.linalg.inv(data['T_cam1_velo']).dot(p_cam)
        p_velo2 = np.linalg.inv(data['T_cam2_velo']).dot(p_cam)
        p_velo3 = np.linalg.inv(data['T_cam3_velo']).dot(p_cam)

        data['b_gray'] = np.linalg.norm(p_velo1 - p_velo0)  # gray baseline
        data['b_rgb'] = np.linalg.norm(p_velo3 - p_velo2)   # rgb baseline

        calib = namedtuple('CalibData', data.keys())(*data.values())
        return data

In [ ]:
def read_calib_file(filepath):
    """Read in a calibration file and parse into a dictionary."""
    data = {}

    with open(filepath, 'r') as f:
        for line in f.readlines():
            key, value = line.split(':', 1)
            # The only non-float values in these files are dates, which
            # we don't care about anyway
            try:
                data[key] = np.array([float(x) for x in value.split()])
            except ValueError:
                pass

    return data

## Dataset

In [ ]:
sem_cfg = yaml.load(open(CONFIG_PATH,'r'), Loader=yaml.SafeLoader)
class_names = sem_cfg['labels']
learning_map = sem_cfg['learning_map']
learning_map_inv = sem_cfg['learning_map_inv']
learning_ignore = sem_cfg['learning_ignore']
sem_color_map = sem_cfg['color_map']

In [ ]:
class SemKittiDataset(Dataset):
  def __init__(self, split = "train", npoints = 50000):
    self.split = split
    self.npoints = npoints
    train_sequences = ['00','01','02','03','04','06','07','09','10']
    val_sequence = '08'
    test_sequence = '05'
    root = os.path.join(DATA_ROOT, "")
    self.calib_root = CALIB_ROOT

    train_pcs = []
    train_labels = []
    train_images = []
    for seq in train_sequences:
      train_pcs += sorted(glob.glob(root+seq+'/velodyne/*.bin'))
      train_labels += sorted(glob.glob(root+seq+'/labels/*.label'))
      train_images += sorted(glob.glob(root+seq+'/image_2/*')) 

    val_pcs = sorted(glob.glob(root+val_sequence+'/velodyne/*.bin'))
    val_labels = sorted(glob.glob(root+val_sequence+'/labels/*.label'))
    val_images = sorted(glob.glob(root+val_sequence+'/image_2/*'))
    
    test_pcs = sorted(glob.glob(root+test_sequence+'/velodyne/*.bin'))
    test_labels = sorted(glob.glob(root+test_sequence+'/labels/*.label'))
    test_images = sorted(glob.glob(root+test_sequence+'/image_2/*'))

    if split == "train":
      self.ids = train_pcs
      self.labels = train_labels
      self.images = train_images
    elif split == "val":
      self.ids = val_pcs
      self.labels = val_labels
      self.images = val_images
    else:
      self.ids = test_pcs
      self.labels = test_labels
      self.images = test_images

  def pcd_normalize(self, pcd):
    pcd = pcd.copy()
    pcd[:,0] = pcd[:,0] / 70
    pcd[:,1] = pcd[:,1] / 70
    pcd[:,2] = pcd[:,2] / 3
    pcd = np.clip(pcd,-1,1)
    return pcd

  def __len__(self):
    return len(self.ids)

  def __getitem__(self, index):
    points = np.fromfile(self.ids[index], dtype=np.float32).reshape(-1, 4)[:,:3]
    raw_label = np.fromfile(self.labels[index], dtype=np.uint32).reshape((-1))
    
    seq = os.path.basename(os.path.dirname(os.path.dirname(self.ids[index])))  # e.g. '08'
    self.calib = load_calib(os.path.join(self.calib_root, seq))
    fn_frame = self.images[index]
    self.frame_BGR = cv2.imread(fn_frame)
    self.frame = cv2.cvtColor(self.frame_BGR, cv2.COLOR_BGR2RGB)
    
    #get labels and map it to classes
    semantic_label = raw_label & 0xFFFF
    mapped_label = np.array([learning_map[x] for x in semantic_label], dtype=np.int64)
    point_colors = np.array([sem_color_map[x] for x in semantic_label], dtype=np.int64)

    # drop points of class zero for unlabeled points, outliers, ...
    drop_class_0 = np.where(mapped_label != 0)
    points = points[drop_class_0]
    mapped_label = mapped_label[drop_class_0] - 1
    point_colors = point_colors[drop_class_0] - 1

    #keep only points that fall on the image (same points as the painted model);
    #the RGB values are computed but not returned / used by this baseline
    points_2d, depth_outlier, size_outlier = self.project_3d_to_2d(points)
    #filter outliers
    points = np.delete(points,depth_outlier,axis=0)
    points = np.delete(points,np.where(size_outlier),axis=0)
    mapped_label = np.delete(mapped_label,depth_outlier,axis=0)
    mapped_label = np.delete(mapped_label,np.where(size_outlier),axis=0)
    point_colors = np.delete(point_colors,depth_outlier,axis=0)
    point_colors = np.delete(point_colors,np.where(size_outlier),axis=0)

    rgb_values = np.zeros((points_2d.shape[0],3), dtype=np.uint8)
    for i in range(points_2d.shape[0]):
      rgb_values[i,:] = self.frame[int(points_2d[i,1]), int(points_2d[i,0])]

    #choose a sample of the points
    length = points.shape[0]
    choice = np.random.choice(length, self.npoints, replace=True)
    points = points[choice]
    label = mapped_label[choice]
    rgb_values = rgb_values[choice]
    point_colors = point_colors[choice]
    points_2d = points_2d[choice]

    #normalize points
    points = self.pcd_normalize(points)                    

    #return {'points': points, 'label', label, 'rgb_values': rgb_values, 'point_colors': point_colors, 'points_2d': points_2d}

    return {'points': points, 'label' : label }

  def project_3d_to_2d(self, pts_3d):
    one_mat = np.ones((pts_3d.shape[0], 1),dtype=np.float32)
    velo = np.concatenate((pts_3d, one_mat), axis=1).T
    R_rect_eye = np.matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
    cam = self.calib['P_rect_20'] * R_rect_eye *self.calib['T_cam2_velo'] * velo
    depth_outlier = np.where(cam[2,:]<0)[1]
    cam = np.delete(cam,depth_outlier,axis=1)
    cam[:2] /= cam[2,:]

    u,v,z = cam
    IMG_H,IMG_W,_ = self.frame.shape
    u_out = np.logical_or(u<0, u>IMG_W)
    v_out = np.logical_or(v<0, v>IMG_H)
    size_outlier = np.logical_or(u_out, v_out)
    cam = np.delete(cam,np.where(size_outlier),axis=1)

    return cam[:2,:].T, depth_outlier, size_outlier

In [ ]:
train_dataset = SemKittiDataset(split = "train", npoints = 10000)
val_dataset = SemKittiDataset(split = "val", npoints = 10000)
test_dataset = SemKittiDataset(split = "test", npoints = 10000)

In [ ]:
print(train_dataset.__len__())
print(val_dataset.__len__())
print(test_dataset.__len__())

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8 , shuffle= True)
val_loader = DataLoader(val_dataset, batch_size=8 , shuffle= False)
sample = next(iter(train_loader))

In [ ]:
print(sample['points'].shape)
print(sample['label'].shape)

## Model
3D-GCN blocks (deformable 3D kernels) and the segmentation network.

In [ ]:
# 3D-GCN building blocks, taken from the official 3D-GCN implementation:
# https://github.com/zhihao-lin/3dgcn  (Lin et al., CVPR 2020)
"""
@Author: Zhi-Hao Lin
@Contact: r08942062@ntu.edu.tw
@Time: 2020/03/06
@Document: Basic operation/blocks of 3D-GCN
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def get_neighbor_index(vertices: "(bs, vertice_num, 3)",  neighbor_num: int):
    """
    Return: (bs, vertice_num, neighbor_num)
    """
    bs, v, _ = vertices.size()
    device = vertices.device
    inner = torch.bmm(vertices, vertices.transpose(1, 2)) #(bs, v, v)
    quadratic = torch.sum(vertices**2, dim= 2) #(bs, v)
    distance = inner * (-2) + quadratic.unsqueeze(1) + quadratic.unsqueeze(2)
    neighbor_index = torch.topk(distance, k= neighbor_num + 1, dim= -1, largest= False)[1]
    neighbor_index = neighbor_index[:, :, 1:]
    return neighbor_index

def get_nearest_index(target: "(bs, v1, 3)", source: "(bs, v2, 3)"):
    """
    Return: (bs, v1, 1)
    """
    inner = torch.bmm(target, source.transpose(1, 2)) #(bs, v1, v2)
    s_norm_2 = torch.sum(source ** 2, dim= 2) #(bs, v2)
    t_norm_2 = torch.sum(target ** 2, dim= 2) #(bs, v1)
    d_norm_2 = s_norm_2.unsqueeze(1) + t_norm_2.unsqueeze(2) - 2 * inner
    nearest_index = torch.topk(d_norm_2, k= 1, dim= -1, largest= False)[1]
    return nearest_index

def indexing_neighbor(tensor: "(bs, vertice_num, dim)", index: "(bs, vertice_num, neighbor_num)" ):
    """
    Return: (bs, vertice_num, neighbor_num, dim)
    """
    bs, v, n = index.size()
    id_0 = torch.arange(bs).view(-1, 1, 1)
    tensor_indexed = tensor[id_0, index]
    return tensor_indexed

def get_neighbor_direction_norm(vertices: "(bs, vertice_num, 3)", neighbor_index: "(bs, vertice_num, neighbor_num)"):
    """
    Return: (bs, vertice_num, neighobr_num, 3)
    """
    neighbors = indexing_neighbor(vertices, neighbor_index) # (bs, v, n, 3)
    neighbor_direction = neighbors - vertices.unsqueeze(2)
    neighbor_direction_norm = F.normalize(neighbor_direction, dim= -1)
    return neighbor_direction_norm

class Conv_surface(nn.Module):
    """Extract structure feafure from surface, independent from vertice coordinates"""
    def __init__(self, kernel_num, support_num):
        super().__init__()
        self.kernel_num = kernel_num
        self.support_num = support_num

        self.relu = nn.ReLU(inplace= True)
        self.directions = nn.Parameter(torch.FloatTensor(3, support_num * kernel_num))
        self.initialize()

    def initialize(self):
        stdv = 1. / math.sqrt(self.support_num * self.kernel_num)
        self.directions.data.uniform_(-stdv, stdv)
    
    def forward(self, 
                neighbor_index: "(bs, vertice_num, neighbor_num)", 
                vertices: "(bs, vertice_num, 3)"):
        """
        Return vertices with local feature: (bs, vertice_num, kernel_num)
        """
        bs, vertice_num, neighbor_num = neighbor_index.size() 
        neighbor_direction_norm = get_neighbor_direction_norm(vertices, neighbor_index)
        support_direction_norm = F.normalize(self.directions, dim= 0) #(3, s * k)
        theta = neighbor_direction_norm @ support_direction_norm # (bs, vertice_num, neighbor_num, s*k)

        theta = self.relu(theta)
        theta = theta.contiguous().view(bs, vertice_num, neighbor_num, self.support_num, self.kernel_num)
        theta = torch.max(theta, dim= 2)[0] # (bs, vertice_num, support_num, kernel_num)
        feature = torch.sum(theta, dim= 2) # (bs, vertice_num, kernel_num)
        return feature

class Conv_layer(nn.Module):
    def __init__(self, in_channel, out_channel, support_num):
        super().__init__()
        # arguments: 
        self.in_channel = in_channel
        self.out_channel = out_channel
        self.support_num = support_num

        # parameters:
        self.relu = nn.ReLU(inplace= True)
        self.weights = nn.Parameter(torch.FloatTensor(in_channel, (support_num + 1) * out_channel))
        self.bias = nn.Parameter(torch.FloatTensor((support_num + 1) * out_channel))
        self.directions = nn.Parameter(torch.FloatTensor(3, support_num * out_channel))
        self.initialize()

    def initialize(self):
        stdv = 1. / math.sqrt(self.out_channel * (self.support_num + 1))
        self.weights.data.uniform_(-stdv, stdv)
        self.bias.data.uniform_(-stdv, stdv)
        self.directions.data.uniform_(-stdv, stdv)

    def forward(self, 
                neighbor_index: "(bs, vertice_num, neighbor_index)",
                vertices: "(bs, vertice_num, 3)",
                feature_map: "(bs, vertice_num, in_channel)"):
        """
        Return: output feature map: (bs, vertice_num, out_channel)
        """
        bs, vertice_num, neighbor_num = neighbor_index.size()
        neighbor_direction_norm = get_neighbor_direction_norm(vertices, neighbor_index)
        support_direction_norm = F.normalize(self.directions, dim= 0)
        theta = neighbor_direction_norm @ support_direction_norm # (bs, vertice_num, neighbor_num, support_num * out_channel)
        theta = self.relu(theta)
        theta = theta.contiguous().view(bs, vertice_num, neighbor_num, -1)
        # (bs, vertice_num, neighbor_num, support_num * out_channel)

        feature_out = feature_map @ self.weights + self.bias # (bs, vertice_num, (support_num + 1) * out_channel)
        feature_center = feature_out[:, :, :self.out_channel] # (bs, vertice_num, out_channel)
        feature_support = feature_out[:, :, self.out_channel:] #(bs, vertice_num, support_num * out_channel)

        # Fuse together - max among product
        feature_support = indexing_neighbor(feature_support, neighbor_index) # (bs, vertice_num, neighbor_num, support_num * out_channel)
        activation_support = theta * feature_support # (bs, vertice_num, neighbor_num, support_num * out_channel)
        activation_support = activation_support.view(bs,vertice_num, neighbor_num, self.support_num, self.out_channel)
        activation_support = torch.max(activation_support, dim= 2)[0] # (bs, vertice_num, support_num, out_channel)
        activation_support = torch.sum(activation_support, dim= 2)    # (bs, vertice_num, out_channel)
        feature_fuse = feature_center + activation_support # (bs, vertice_num, out_channel)
        return feature_fuse

class Pool_layer(nn.Module):
    def __init__(self, pooling_rate: int= 4, neighbor_num: int=  4):
        super().__init__()
        self.pooling_rate = pooling_rate
        self.neighbor_num = neighbor_num

    def forward(self, 
                vertices: "(bs, vertice_num, 3)",
                feature_map: "(bs, vertice_num, channel_num)"):
        """
        Return:
            vertices_pool: (bs, pool_vertice_num, 3),
            feature_map_pool: (bs, pool_vertice_num, channel_num)
        """
        bs, vertice_num, _ = vertices.size()
        neighbor_index = get_neighbor_index(vertices, self.neighbor_num)
        neighbor_feature = indexing_neighbor(feature_map, neighbor_index) #(bs, vertice_num, neighbor_num, channel_num)
        pooled_feature = torch.max(neighbor_feature, dim= 2)[0] #(bs, vertice_num, channel_num)

        pool_num = int(vertice_num / self.pooling_rate)
        sample_idx = torch.randperm(vertice_num)[:pool_num]
        vertices_pool = vertices[:, sample_idx, :] # (bs, pool_num, 3)
        feature_map_pool = pooled_feature[:, sample_idx, :] #(bs, pool_num, channel_num)
        return vertices_pool, feature_map_pool

def test():
    import time
    bs = 8
    v = 1024
    dim = 3
    n = 20
    vertices = torch.randn(bs, v, dim)
    neighbor_index = get_neighbor_index(vertices, n)

    s = 3
    conv_1 = Conv_surface(kernel_num= 32, support_num= s)
    conv_2 = Conv_layer(in_channel= 32, out_channel= 64, support_num= s)
    pool = Pool_layer(pooling_rate= 4, neighbor_num= 4)
    
    print("Input size: {}".format(vertices.size()))
    start = time.time()
    f1 = conv_1(neighbor_index, vertices)
    print("\n[1] Time: {}".format(time.time() - start))
    print("[1] Out shape: {}".format(f1.size()))
    start = time.time()
    f2 = conv_2(neighbor_index, vertices, f1)
    print("\n[2] Time: {}".format(time.time() - start))
    print("[2] Out shape: {}".format(f2.size()))
    start = time.time()
    v_pool, f_pool = pool(vertices, f2)
    print("\n[3] Time: {}".format(time.time() - start))
    print("[3] v shape: {}, f shape: {}".format(v_pool.size(), f_pool.size()))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys

class GCN3D(nn.Module):
    def __init__(self, class_num, support_num, neighbor_num):
        super().__init__()
        self.neighbor_num = neighbor_num

        self.conv_0 = Conv_surface(kernel_num= 128, support_num= support_num)
        self.conv_1 = Conv_layer(128, 128, support_num= support_num)
        self.pool_1 = Pool_layer(pooling_rate= 4, neighbor_num= 4)
        self.conv_2 = Conv_layer(128, 256, support_num= support_num)
        self.conv_3 = Conv_layer(256, 256, support_num= support_num)
        self.pool_2 = Pool_layer(pooling_rate= 4, neighbor_num= 4)
        self.conv_4 = Conv_layer(256, 512, support_num= support_num)

        ####remove 16 as it is the dimension of the one hot encoded category
        dim_fuse = sum([128, 128, 256, 256, 512, 512])
        #####
        self.conv1d_block = nn.Sequential(
            nn.Conv1d(dim_fuse, 512, 1),
            nn.ReLU(inplace= True),
            nn.Conv1d(512, 512, 1),
            nn.ReLU(inplace= True),
            nn.Conv1d(512, class_num, 1),
        )

    def forward(self, 
                vertices: "tensor (bs, vetice_num, 3)" 
                ):
        """
        Return: (bs, vertice_num, class_num)
        """

        bs, vertice_num, _ = vertices.shape
        neighbor_index = get_neighbor_index(vertices, self.neighbor_num)

        fm_0 = F.relu(self.conv_0(neighbor_index, vertices), inplace= True)
        fm_1 = F.relu(self.conv_1(neighbor_index, vertices, fm_0), inplace= True)
        v_pool_1, fm_pool_1 = self.pool_1(vertices, fm_1)
        neighbor_index = get_neighbor_index(v_pool_1, self.neighbor_num)

        fm_2 = F.relu(self.conv_2(neighbor_index, v_pool_1, fm_pool_1), inplace= True)
        fm_3 = F.relu(self.conv_3(neighbor_index, v_pool_1, fm_2), inplace= True)
        v_pool_2, fm_pool_2 = self.pool_2(v_pool_1, fm_3)
        neighbor_index = get_neighbor_index(v_pool_2, self.neighbor_num)

        fm_4 = self.conv_4(neighbor_index, v_pool_2, fm_pool_2)
        f_global = fm_4.max(1)[0] #(bs, f)

        nearest_pool_1 = get_nearest_index(vertices, v_pool_1)
        nearest_pool_2 = get_nearest_index(vertices, v_pool_2)
        fm_2 = indexing_neighbor(fm_2, nearest_pool_1).squeeze(2)
        fm_3 = indexing_neighbor(fm_3, nearest_pool_1).squeeze(2)
        fm_4 = indexing_neighbor(fm_4, nearest_pool_2).squeeze(2)
        f_global = f_global.unsqueeze(1).repeat(1, vertice_num, 1)
        # replace by 
        fm_fuse = torch.cat([fm_0, fm_1, fm_2, fm_3, fm_4, f_global], dim= 2)
        #####
        conv1d_input = fm_fuse.permute(0, 2, 1) #(bs, fuse_ch, vertice_num)
        conv1d_out = self.conv1d_block(conv1d_input) 
        pred = conv1d_out.permute(0, 2, 1) #(bs, vertice_num, ch)
        return pred

In [ ]:
model = GCN3D(class_num= 19, support_num= 1, neighbor_num= 50)
print(model)

## Training

In [ ]:
import os
import sys
#from util import *
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
#from visualize import visualize

class Manager():
    def __init__(self, model, args):
        self.args_info = args.__str__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        n_gpu = torch.cuda.device_count()
        print(f'n_gpu = {n_gpu}')
        device_ids = list(range(n_gpu))
        print(f"lr = {args['lr']}")
        self.model = model.to(self.device)
        self.model = torch.nn.DataParallel(model, device_ids=device_ids)
        
        if args['load']:
            self.model.load_state_dict(torch.load(args['load'], map_location=self.device))
        self.epoch = args['epoch']
        self.optimizer = optim.Adam(self.model.parameters(), lr= args['lr'])
        self.lr_scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size= 10, gamma= 0.5)
        self.loss_function = nn.CrossEntropyLoss()

        self.chkpt_dir = os.path.join(OUTPUT_DIR, "checkpoints")
        self.model_path = args['model_path']
        self.save = os.path.join(OUTPUT_DIR, "models", args['model_path'] + ".pt")
        self.record_interval = args['interval']
        self.record_file = None
        if args['record']:
            self.record_file = open(args['record'], 'w')
        
        self.out_dir = args['output']
        self.best = {"c_miou": args['best_c_miou'], "i_miou": args['best_c_miou']}
        self.start = args['start']
        
        """adding comet experiment"""
        hyper_params = {
            "backbone": "initial",
            "learning_rate": args['lr'],
            "batchSize": args['bs'],
            "optimizer": "Adam",
            "npoints": args["npoints"]
        }
        
        # Comet ML logging is optional: set COMET_API_KEY (and COMET_WORKSPACE) to enable it
        api_key = os.environ.get("COMET_API_KEY")
        self.experiment = Experiment(
            api_key=api_key,
            project_name="sdc",
            workspace=os.environ.get("COMET_WORKSPACE"),
            disabled=api_key is None,
        )
        self.experiment.set_name(args['model_path'])
        self.experiment.log_parameters(hyper_params)
        """adding comet experiment"""
        self.best_c_miou = args['best_c_miou']
        
    def update_best(self, c_miou, i_miou):
        self.best["c_miou"] = max(self.best["c_miou"], c_miou)
        self.best["i_miou"] = max(self.best["i_miou"], i_miou)

    def record(self, info):
        print(info)
        if self.record_file:
            self.record_file.write(info + '\n')

    """changed function"""
    def calculate_save_mious(self, iou_table, target, pred_choice):
        num_classes = 19
        for class_id in range(num_classes):
          I = torch.sum((pred_choice == class_id) & (target == class_id)).cpu().item()
          U = torch.sum((pred_choice == class_id) | (target == class_id)).cpu().item()
          iou = 1 if U == 0 else I/U
          iou_table.add_obj_miou(class_names[learning_map_inv[class_id+1]], iou)
            

    def save_visualizations(self, dir, category_names, object_ids, points, labels, predictions):
        for i in range(len(category_names)):
            cat = category_names[i]
            valid_labels = get_valid_labels(cat)
            shift = min(valid_labels) * (-1)
            obj_id = object_ids[i]
            point = points[i].to("cpu") 
            label = labels[i].to("cpu") + shift
            pred  = predictions[i].to("cpu") + shift

            cat_dir = os.path.join(dir, cat)
            if not os.path.isdir(cat_dir):
                os.mkdir(cat_dir)
            gt_fig_name = os.path.join(cat_dir, "{}_gt.png".format(obj_id))        
            pred_fig_name = os.path.join(cat_dir, "{}_pred.png".format(obj_id)) 
            visualize(point, label, gt_fig_name)
            visualize(point, pred, pred_fig_name)

    def train(self, train_data, test_data):

        for epoch in range(self.start, self.epoch):
            self.model.train()
            train_loss = 0
            train_iou_table = IouTable()
            for i, batch in enumerate(train_data):
                points = batch['points']
                labels = batch['label']
                points = points.to(self.device)
                labels = labels.to(self.device)
                #to receive in point painting
                #onehot = onehot.to(self.device)
                out = self.model(points)

                self.optimizer.zero_grad()
                loss = self.loss_function(out.reshape(-1, out.size(-1)), labels.view(-1,))     
                loss.backward()
                self.optimizer.step()
                train_loss += loss.item()
                
                #2 is the dimension of the classes , [1] to get the index of the max not the values
                #pred = torch.max(out, 2)[1]
                pred = out.argmax(-1)
                self.calculate_save_mious(train_iou_table, labels.squeeze(-1), pred)

                #print(f"batch step = {i}, iou = {train_iou_table.get_mean_category_miou()}")
                if self.record_interval and ((i + 1) % self.record_interval == 0):
                    #print("=======================================================")
                    #print(f"after 100 epoch")
                    c_miou = train_iou_table.get_mean_category_miou()
                    #i_miou = train_iou_table.get_mean_instance_miou()
                    self.record(' epoch {:3} step {:5} | avg loss: {:.3f} | miou(c): {:.3f}'.format(epoch+1, i+1, train_loss/(i + 1), c_miou))
                    chkpt_path = os.path.join(self.chkpt_dir, "{}_ep{}_step{}_train-miou{:.3f}.pt".format(self.model_path, epoch+1, i+1, c_miou))
                    torch.save(self.model.state_dict(), chkpt_path)
                    #print("=======================================================")
                
            train_loss /= (i+1)
            
            print(f"train_loss after {epoch} epoch = {train_loss}")
            train_table_str = train_iou_table.get_string()
            train_c_miou = train_iou_table.get_mean_category_miou()
            print(f"train_c_miou = {train_c_miou}")
            
            #del points, labels, out, pred
            #test_loss, test_table_str, val_c_miou = self.test(test_data, self.out_dir)
            if self.out_dir: 
                if not os.path.isdir(self.out_dir):
                    os.mkdir(self.out_dir)

            self.model.eval()
            test_loss = 0
            test_iou_table = IouTable()

            for i, batch in enumerate(test_data):
                points = batch['points']
                labels = batch['label']
                points = points.to(self.device)
                labels = labels.to(self.device)
                with torch.no_grad():
                    out = self.model(points)
                loss = self.loss_function(out.reshape(-1, out.size(-1)), labels.view(-1,))     
                test_loss += loss.item()

                pred = out.argmax(-1)
                self.calculate_save_mious(test_iou_table, labels, pred)

                """to be done later
                if out_dir:
                    self.save_visualizations(out_dir, cat_name, obj_ids, points, labels, pred)
                """

            test_loss /= (i+1) 
            val_c_miou = test_iou_table.get_mean_category_miou()
            i_miou = test_iou_table.get_mean_instance_miou()
            self.update_best(val_c_miou,i_miou)
            test_table_str = test_iou_table.get_string()

            if self.out_dir:
                miou_file = open(os.path.join(self.out_dir, "miou.txt"), "w")
                miou_file.write(test_table_str)
                
            self.lr_scheduler.step()
            print(f"test_table_str = {test_table_str}")
            
            """ added comet logging"""
            loss_dict = {
                "train_loss": train_loss, "val_loss": test_loss }
            metrics_dict = {
                "train_c_miou": train_c_miou, "val_c_miou": val_c_miou }
            self.experiment.log_metrics(loss_dict, epoch=epoch+1)
            self.experiment.log_metrics(metrics_dict, epoch=epoch+1)
            """ adding comet logging"""
            
            if val_c_miou >= self.best_c_miou:
                self.best_c_miou = val_c_miou
                torch.save(self.model.state_dict(), self.save)

            self.record("==== Epoch {:3} ====".format(epoch + 1))
            self.record("Training mIoU:")
            self.record(train_table_str)
            self.record("Testing mIoU:")
            self.record(test_table_str)
            self.record("* Best mIoU(c): {:.3f}, Best mIoU (i): {:.3f} \n".format(self.best["c_miou"], self.best["i_miou"]))

        self.experiment.end()
        """
    def test(self, test_data, out_dir= None):
        
        self.update_best(c_miou,i_miou)
        test_table_str = test_iou_table.get_string()

        if out_dir:
            miou_file = open(os.path.join(out_dir, "miou.txt"), "w")
            miou_file.write(test_table_str)

        return test_loss, test_table_str, c_miou
        
"""

In [ ]:
class IouTable():
    def __init__(self):
        self.obj_miou = {}
        
    def add_obj_miou(self, category: int, miou: float):
        if category not in self.obj_miou:
            self.obj_miou[category] = [miou]
        else:
            self.obj_miou[category].append(miou)

    def get_category_miou(self):
        """
        Return: moiu table of each category
        """
        category_miou = {}
        for c, mious in self.obj_miou.items():
            category_miou[c] = np.mean(mious)
        return category_miou

    def get_mean_category_miou(self):
        category_miou = []
        for c, mious in self.obj_miou.items():
            c_miou = np.mean(mious)
            category_miou.append(c_miou)
        return np.mean(category_miou)
    
    def get_mean_instance_miou(self):
        object_miou = []
        for c, mious in self.obj_miou.items():
            object_miou += mious
        return np.mean(object_miou)

    def get_string(self):
        mean_c_miou = self.get_mean_category_miou()
        mean_i_miou = self.get_mean_instance_miou()
        first_row  = "| {:5} | {:5} ||".format("Avg_c", "Avg_i")
        second_row = "| {:.3f} | {:.3f} ||".format(mean_c_miou, mean_i_miou)
        
        categories = list(self.obj_miou.keys())
        categories.sort()
        cate_miou = self.get_category_miou()
        
        table = ""
        for c in categories:
            miou = cate_miou[c]
            #modified since the categories are int num
            #first_row  += " {:5} |".format(c[:3]])
            ##### for class index
            #first_row  += f" {c} |"
            ##for the string => class_names[learning_map_inv[c]]
            #first_row  += " {:5} |".format(class_names[learning_map_inv[c]])
            #second_row += " {:.3f} |".format(miou)

            """my table"""
            table += f"{c} : {round(miou, 5)} \n" 
        
        avg = first_row + "\n" + second_row
        return avg+"\n"+table  

def test():
    table = IouTable()
    cates = ['car', 'bicycle', 'bus', 'motorcycle', 'on-rails', 'truck']
    for e in range(10):
        for c in cates:
            table.add_obj_miou(c, random.random())
    avg, table = table.get_string()
    print(avg)
    print(table) 

In [ ]:
# Baseline configuration from the report: raw LiDAR points (XYZ), lr = 0.001, batch size 8 -> val mIoU 0.595.
args = {
    "mode": "train",
    "support": 1,
    "neighbor": 50,
    "model_path": "gcn3d_no_rgb_lr0.001_b8",
    "epoch": 100,
    "lr": 0.001,
    "bs": 8,
    "npoints": 10000,
    "record": os.path.join(OUTPUT_DIR, "gcn3d_no_rgb_lr0.001_b8.log"),
    "interval": 1000,
    "load": None,
    "start": 0,
    "best_c_miou": 0.0,
    "output": OUTPUT_DIR,
}
manager = Manager(model, args)

In [ ]:
manager.train(train_loader, val_loader)